# SPFI ×4 — trained width comparison C8 vs C16 on the validation split (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/main/notebooks/02_width_eval_val_colab.ipynb)

Compares two SPFI models that differ **only in width** (`embed_dim` 8 vs 16), trained with
`train_colab.ipynb` under the same budget (same `MAX_ITERS`, seeds, data and settings).

* **Quality:** PSNR / SSIM per image on the **validation split** (the 20% of EMPS `train.csv` images the
  trainer held out, reproduced with `algo.train.split_files`), plus bicubic as a reference.
  **The test split is never downloaded** — it stays locked for the final evaluation.
* **Cost:** parameters, FLOPs, inference latency and peak memory (this GPU), and training time / memory
  from the `train_info_*.json` files written by `train_colab.ipynb`.
* **Paired analysis:** both models see the same images, so the per-image difference C16 − C8 is reported
  with a bootstrap 95% confidence interval.

> **Status: exploratory.** One training seed per width, a fixed short budget, validation split only.
> These numbers are a first look, not the approved comparison (that waits for approved checkpoints, #156).

**Before you start:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.

## Settings
Point `CHECKPOINTS` at the two **final** files from `train_colab.ipynb` (both trained for exactly `MAX_ITERS`).
Leave a path empty to upload the file from your computer. `TRAIN_INFO` is optional (adds training cost).

In [ ]:
import os
from pathlib import Path

# ---- Code ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "main"

# ---- Data: public EMPS release (same as train_colab.ipynb); only train.csv images are extracted ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"
SPLIT_SEED = 0          # must equal SPLIT_SEED used for training -> same validation images

# ---- The two models (label -> checkpoint path). Empty string = upload. ----
CHECKPOINTS = {
    "C8":  "/content/drive/MyDrive/spfi_checkpoints/spfi_x4_final_e8_s0.pt",
    "C16": "/content/drive/MyDrive/spfi_checkpoints/spfi_x4_final_e16_s0.pt",
}
EXPECTED_EMBED_DIM = {"C8": 8, "C16": 16}
TRAIN_INFO = {          # optional; missing files are skipped
    "C8":  "/content/drive/MyDrive/spfi_checkpoints/train_info_e8_s0.json",
    "C16": "/content/drive/MyDrive/spfi_checkpoints/train_info_e16_s0.json",
}

# ---- Evaluation (same tiling as inference_colab.ipynb) ----
TILE = 64            # LR tile size (= training LR patch size)
OVERLAP = 16         # LR pixels shared by neighbouring tiles
BATCH_TILES = 16     # tiles per forward pass
SHAVE = 4            # HR border pixels ignored by PSNR / SSIM
MAX_VAL_IMAGES = None    # None = whole validation split; a number = quick check on the first N
BOOTSTRAP = 10_000   # resamples for the confidence interval of the paired difference
TIMING_RUNS = 50     # forward passes timed per model (after warm-up)

IN_COLAB = "google.colab" in __import__("sys").modules
WORK_DIR = Path("/content") if IN_COLAB else Path(os.environ.get("SR_WORK_DIR", Path.cwd() / "_width_eval"))
CODE_DIR = WORK_DIR / "SR_amat_3"
DATA_DIR = WORK_DIR / "emps"
OUTPUT_DIR = WORK_DIR / "output"
# Local testing only: environment variables override the values above.
CODE_REPO_URL = os.environ.get("SR_REPO_URL") or CODE_REPO_URL
CODE_BRANCH = os.environ.get("SR_REF") or CODE_BRANCH
if os.environ.get("SR_CKPT_C8"):
    CHECKPOINTS = {"C8": os.environ["SR_CKPT_C8"], "C16": os.environ["SR_CKPT_C16"]}
    TRAIN_INFO = {"C8": os.environ.get("SR_INFO_C8", ""), "C16": os.environ.get("SR_INFO_C16", "")}
if os.environ.get("SR_MAX_VAL_IMAGES"):
    MAX_VAL_IMAGES = int(os.environ["SR_MAX_VAL_IMAGES"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Code and dependencies

In [ ]:
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pytorch_wavelets>=1.3", "PyWavelets>=1.6", "setuptools<81"], check=True)

In [ ]:
import hashlib
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout.strip()
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def download_emps():
    """Download the pinned public EMPS archive once (no credentials needed) and check its SHA-256."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    return archive


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_train():
    """Extract train.csv, test.csv and only the train.csv images into DATA_DIR/train/."""
    dest = DATA_DIR / "train"
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_emps()) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / "train.csv").read_text())
        for image_id in ids:
            out = dest / "images" / f"{image_id}.png"
            if not out.exists():
                out.write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", CODE_REPO_URL, str(CODE_DIR))
git("fetch", "--quiet", "origin", CODE_BRANCH, cwd=CODE_DIR)
git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
COMMIT = git("rev-parse", "HEAD", cwd=CODE_DIR)
print("code:", git("log", "-1", "--format=%h %s", cwd=CODE_DIR))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

## Validation images
The trainer splits the `train.csv` images 80/20 with `SPLIT_SEED`; `split_files` reproduces that split.
The test images are not extracted, and the check below makes sure none is used.

In [ ]:
from algo.train import split_files

EMPS_DIR, train_ids = extract_emps_train()
test_ids = set(read_split((EMPS_DIR / "test.csv").read_text()))
on_disk = sorted(p.stem for p in (EMPS_DIR / "images").glob("*.png"))
assert on_disk == sorted(train_ids), "images/ must hold exactly the train split"
assert not set(on_disk) & test_ids, "test images must never be used here"

train_files, val_ids = split_files(on_disk, SPLIT_SEED)
if MAX_VAL_IMAGES is not None:
    val_ids = val_ids[:MAX_VAL_IMAGES]
print(f"{len(on_disk)} train.csv images -> {len(train_files)} used for training / "
      f"{len(val_ids)} validation images evaluated here (split seed {SPLIT_SEED}); test split untouched")

## Load both models
Each checkpoint is loaded the way the local app loads it. The width must match the label, and both must
have the same number of training iterations — otherwise the comparison is not fixed-budget.

In [ ]:
import json
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import torch

from algo.model import build_spfi

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU — quality numbers are still valid, timing is CPU timing.")


def resolve(path, label):
    if path and path.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if path and Path(path).exists():
        return Path(path)
    if IN_COLAB:
        from google.colab import files
        print(f"Upload the {label} file" + (f" ({Path(path).name})" if path else ""))
        name, data = next(iter(files.upload().items()))
        (WORK_DIR / name).write_bytes(data)
        return WORK_DIR / name
    return None


MODELS, META = {}, {}
for label, path in CHECKPOINTS.items():
    p = resolve(path, f"{label} checkpoint")
    assert p is not None, f"{label}: checkpoint not found: {path}"
    ckpt = torch.load(p, map_location=device, weights_only=True)
    hp = ckpt["hparams"]
    model = build_spfi(scale=hp["scale"], embed_dim=hp["embed_dim"])
    model.load_state_dict(ckpt["model_state"])
    MODELS[label] = model.to(device).eval()
    META[label] = {
        "file": p.name,
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        "embed_dim": hp["embed_dim"],
        "scale": hp["scale"],
        "iteration": ckpt.get("iteration"),
        "params": sum(q.numel() for q in model.parameters()),
    }
    print(f"{label}: {p.name}  embed_dim={hp['embed_dim']}  iteration={ckpt.get('iteration')}  "
          f"params={META[label]['params']:,}  sha256={META[label]['sha256'][:12]}…")

SCALE = META["C8"]["scale"]
for label, m in META.items():
    assert m["embed_dim"] == EXPECTED_EMBED_DIM[label], f"{label} has embed_dim {m['embed_dim']}"
    assert m["scale"] == SCALE == 4
SAME_BUDGET = META["C8"]["iteration"] == META["C16"]["iteration"]
print("same training iterations:", SAME_BUDGET,
      "" if SAME_BUDGET else "  <-- NOT a fixed-budget comparison; interpret with care")

TRAIN_INFOS = {}
for label, path in TRAIN_INFO.items():
    if path and path.startswith("/content/drive") and IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    if path and Path(path).exists():
        TRAIN_INFOS[label] = json.loads(Path(path).read_text())
print("training info:", ", ".join(TRAIN_INFOS) or "none (training cost columns will be empty)")

## Quality on the validation split
Same procedure as `inference_colab.ipynb`: degrade each HR image with the repo's `degrade`, super-resolve
with overlapping 64×64 tiles, compare to HR with `psnr` / `ssim_loss` from `algo.train`, ignoring a
`SHAVE`-pixel border. Bicubic upscaling of the same LR image is the reference.

In [ ]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm

from algo.image_loading import load_image
from algo.preprocessing import normalize_image
from algo.train import degrade, psnr, ssim_loss


def tile_starts(n):
    """Start offsets of TILE-sized windows covering 0..n, OVERLAP apart, the last one flush with the end."""
    if n <= TILE:
        return [0]
    starts = list(range(0, n - TILE, TILE - OVERLAP))
    return starts + [n - TILE]


@torch.no_grad()
def super_resolve(model, lr):
    """[1, 1, h, w] LR -> [1, 1, h*SCALE, w*SCALE] SR, tiled exactly as in inference_colab.ipynb."""
    _, _, h, w = lr.shape
    lr = F.pad(lr, (0, max(0, TILE - w), 0, max(0, TILE - h)), mode="replicate")
    H, W = lr.shape[-2:]
    coords = [(y, x) for y in tile_starts(H) for x in tile_starts(W)]
    out = torch.zeros(1, 1, H * SCALE, W * SCALE, device=lr.device)
    hits = torch.zeros_like(out)
    t = TILE * SCALE
    for i in range(0, len(coords), BATCH_TILES):
        chunk = coords[i:i + BATCH_TILES]
        sr = model(torch.cat([lr[..., y:y + TILE, x:x + TILE] for y, x in chunk]))
        for (y, x), tile in zip(chunk, sr):
            out[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += tile
            hits[..., y * SCALE:y * SCALE + t, x * SCALE:x * SCALE + t] += 1
    return (out / hits)[..., :h * SCALE, :w * SCALE]


def shave(img):
    return img[..., SHAVE:-SHAVE, SHAVE:-SHAVE] if SHAVE else img


rows = []
for image_id in tqdm(val_ids):
    hr = to_tensor(normalize_image(load_image(EMPS_DIR / "images" / f"{image_id}.png")))   # [1, H, W]
    _, H, W = hr.shape
    hr = hr[:, :H - H % SCALE, :W - W % SCALE]
    lr = degrade(hr, filename=image_id, scale=SCALE)
    hr4 = shave(hr.unsqueeze(0))
    bicubic = F.interpolate(lr.unsqueeze(0), scale_factor=SCALE, mode="bicubic", align_corners=False).clamp(0, 1)
    row = {"image": image_id, "hr_size": f"{hr.shape[-1]}x{hr.shape[-2]}",
           "bicubic_psnr": psnr(shave(bicubic), hr4),
           "bicubic_ssim": 1.0 - ssim_loss(shave(bicubic), hr4).item()}
    for label, model in MODELS.items():
        sr = super_resolve(model, lr.unsqueeze(0).to(device)).clamp(0, 1).cpu()
        row[f"{label}_psnr"] = psnr(shave(sr), hr4)
        row[f"{label}_ssim"] = 1.0 - ssim_loss(shave(sr), hr4).item()
    rows.append(row)

results = pd.DataFrame(rows)
results["delta_psnr"] = results["C16_psnr"] - results["C8_psnr"]
results["delta_ssim"] = results["C16_ssim"] - results["C8_ssim"]
results.to_csv(OUTPUT_DIR / "val_results.csv", index=False)
results.head()

## Paired difference C16 − C8
Both models were evaluated on the same images, so the per-image difference removes image difficulty.
The 95% interval comes from resampling images with replacement (bootstrap). If it includes 0, this
validation set cannot tell the two widths apart at this budget.

In [ ]:
rng = np.random.default_rng(0)


def paired_summary(delta):
    delta = np.asarray(delta, dtype=float)
    boots = rng.choice(delta, size=(BOOTSTRAP, len(delta)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"mean": delta.mean(), "ci95_low": lo, "ci95_high": hi,
            "C16_better": int((delta > 0).sum()), "C8_better": int((delta < 0).sum()), "n": len(delta)}


PAIRED = {"PSNR (dB)": paired_summary(results["delta_psnr"]),
          "SSIM": paired_summary(results["delta_ssim"])}
paired_table = pd.DataFrame(PAIRED).T
paired_table["C16 better on"] = [f"{int(r['C16_better'])}/{int(r['n'])} images" for _, r in paired_table.iterrows()]
display(paired_table[["mean", "ci95_low", "ci95_high", "C16 better on"]].astype({"mean": float}).round(4))

## Cost
* **Params / FLOPs:** per 64×64 LR tile (256×256 output).
* **Latency / memory:** median time of one forward pass on a batch of `BATCH_TILES` tiles after warm-up,
  and peak GPU memory of that pass, on **this** GPU. Re-measure on the same GPU before comparing to other runs.
* **Training:** from `train_info_*.json` (time, seconds per iteration, peak memory), if provided.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


@torch.no_grad()
def measure(model):
    x = torch.rand(BATCH_TILES, 1, TILE, TILE, device=device)
    with FlopCounterMode(display=False) as counter:
        model(x[:1])
    gflops = counter.get_total_flops() / 1e9
    for _ in range(5):
        model(x)
    times = []
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    for _ in range(TIMING_RUNS):
        t0 = time.perf_counter()
        model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    peak = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else None
    return {"gflops_per_tile": gflops, "latency_ms_per_batch": 1000 * float(np.median(times)), "peak_infer_mem_mb": peak}


COST = {}
for label, model in MODELS.items():
    info = TRAIN_INFOS.get(label, {})
    COST[label] = {
        "params": META[label]["params"],
        **measure(model),
        "train_minutes": info.get("train_seconds", np.nan) / 60 if info else np.nan,
        "train_s_per_iter": info.get("seconds_per_iter", np.nan),
        "train_peak_mem_mb": info.get("peak_gpu_memory_mb", np.nan),
        "train_gpu": info.get("gpu"),
    }
cost_table = pd.DataFrame(COST).T
cost_table.loc["C16 / C8"] = [
    (cost_table.loc["C16", c] / cost_table.loc["C8", c]) if isinstance(cost_table.loc["C8", c], (int, float, np.floating)) and cost_table.loc["C8", c] else None
    for c in cost_table.columns]
print("timing device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU",
      f"| batch {BATCH_TILES}×{TILE}×{TILE}")
display(cost_table)

## Summary table and plots

In [ ]:
import matplotlib.pyplot as plt

summary = pd.DataFrame({
    label: {
        "embed_dim": META[label]["embed_dim"] if label in META else "—",
        "iterations": META[label]["iteration"] if label in META else "—",
        "params": f"{META[label]['params']:,}" if label in META else "—",
        "mean PSNR (dB)": round(results[f"{key}_psnr"].mean(), 3),
        "mean SSIM": round(results[f"{key}_ssim"].mean(), 4),
        "PSNR gain vs bicubic (dB)": round((results[f"{key}_psnr"] - results["bicubic_psnr"]).mean(), 3),
    }
    for label, key in (("bicubic", "bicubic"), ("C8", "C8"), ("C16", "C16"))
}).T
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ["bicubic", "C8", "C16"]
colors = ["#9AA0A6", "#4C72B0", "#DD8452"]
for ax, metric, unit in ((axes[0], "psnr", "dB"), (axes[1], "ssim", "")):
    means = [results[f"{k}_{metric}"].mean() for k in labels]
    bars = ax.bar(labels, means, color=colors, width=0.55)
    ax.bar_label(bars, labels=[f"{m:.3f}" if metric == "psnr" else f"{m:.4f}" for m in means], padding=3)
    lo = min(means) - (0.5 if metric == "psnr" else 0.02)
    ax.set_ylim(lo, max(means) + (0.4 if metric == "psnr" else 0.015))
    ax.set_title(f"mean {metric.upper()} on {len(results)} val images")
    ax.set_ylabel(unit or metric.upper())
    ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
ax.scatter(results["C8_psnr"], results["C16_psnr"], s=14, alpha=0.7, color="#DD8452")
lims = [min(results[["C8_psnr", "C16_psnr"]].min()), max(results[["C8_psnr", "C16_psnr"]].max())]
ax.plot(lims, lims, color="#5F6368", lw=1, ls="--")
ax.set_xlabel("C8 PSNR (dB)")
ax.set_ylabel("C16 PSNR (dB)")
ax.set_title("per image (above the line = C16 better)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "width_comparison.png", dpi=130)
plt.show()

## Save and download

In [ ]:
import platform

REPORT = {
    "status": "exploratory (validation split, one seed per width)",
    "code_branch": CODE_BRANCH,
    "code_commit": COMMIT,
    "split_seed": SPLIT_SEED,
    "n_val_images": len(results),
    "checkpoints": META,
    "same_training_iterations": SAME_BUDGET,
    "train_info": TRAIN_INFOS,
    "quality": summary.to_dict(orient="index"),
    "paired_C16_minus_C8": PAIRED,
    "cost": {k: {kk: (None if isinstance(vv, float) and np.isnan(vv) else vv) for kk, vv in v.items()}
             for k, v in COST.items()},
    "env": {"python": platform.python_version(), "torch": torch.__version__,
            "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"},
}
report_path = OUTPUT_DIR / "width_comparison_report.json"
report_path.write_text(json.dumps(REPORT, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("saved:", report_path.name, "val_results.csv", "width_comparison.png")
if IN_COLAB:
    from google.colab import files
    for f in (report_path, OUTPUT_DIR / "val_results.csv", OUTPUT_DIR / "width_comparison.png"):
        files.download(str(f))

## Conclusion

In [ ]:
p, s = PAIRED["PSNR (dB)"], PAIRED["SSIM"]
c8, c16 = COST["C8"], COST["C16"]
verdict = ("C16 is better" if p["ci95_low"] > 0 else
           "C8 is better" if p["ci95_high"] < 0 else
           "no clear PSNR difference")
print(f"Validation split, {len(results)} images, {META['C8']['iteration']} vs {META['C16']['iteration']} training iterations, split seed {SPLIT_SEED}.")
print(f"PSNR C16 − C8: {p['mean']:+.3f} dB (95% CI {p['ci95_low']:+.3f} … {p['ci95_high']:+.3f}); "
      f"C16 better on {p['C16_better']}/{p['n']} images  ->  {verdict} at this budget.")
print(f"SSIM C16 − C8: {s['mean']:+.4f} (95% CI {s['ci95_low']:+.4f} … {s['ci95_high']:+.4f}).")
print(f"Cost C16 / C8: params ×{c16['params'] / c8['params']:.2f}, FLOPs ×{c16['gflops_per_tile'] / c8['gflops_per_tile']:.2f}, "
      f"latency ×{c16['latency_ms_per_batch'] / c8['latency_ms_per_batch']:.2f}"
      + (f", train time ×{c16['train_minutes'] / c8['train_minutes']:.2f}" if not np.isnan(c8['train_minutes']) else ""))

**Limitations:** one training seed per width (seed-to-seed variation is not measured yet — train a second
seed with the same `SPLIT_SEED` before drawing conclusions); a fixed short budget (the wider model may need
more iterations to converge, so a small gap here may change with longer training); validation split only;
timings depend on the GPU and are only comparable within one session.